In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
torch.device

torch.device

In [3]:
net = nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))

X=torch.rand(2,20)

net(X)

tensor([[-0.0109, -0.0414, -0.0756,  0.1755, -0.1016, -0.3861, -0.2242,  0.1660,
         -0.1130, -0.1452],
        [ 0.0128, -0.1695,  0.1095,  0.2581, -0.0804, -0.3731, -0.2413,  0.1323,
         -0.1929, -0.1436]], grad_fn=<AddmmBackward0>)

In [4]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20,256)
        self.out = nn.Linear(256,10)

    def forward(self,X):
        return self.out(F.relu(self.hidden(X)))


In [5]:
class MySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()
        for i,block in enumerate(args):
            self.add_module(str(i), block)
    def forward(self,X):
        for block in self._modules.values():
            X=block(X)
        return X

In [6]:
net=MySequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net(X)

tensor([[ 0.0525,  0.0820,  0.0452, -0.0167,  0.0329,  0.1449,  0.1824,  0.0729,
         -0.0178, -0.0942],
        [-0.1464,  0.1355,  0.0112,  0.1221, -0.1056,  0.0227,  0.0853, -0.0528,
         -0.0651, -0.1796]], grad_fn=<AddmmBackward0>)

In [7]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(20,64),nn.ReLU(),nn.Linear(64,32))
        self.linear=nn.Linear(32,16)

    def forward(self,X):
        return self.linear(self.net(X))


cincc=MySequential(NestMLP(),nn.Linear(16,20),NestMLP())
cincc(X)

tensor([[-0.0061, -0.0269,  0.0565, -0.1135,  0.0231,  0.1206, -0.0217,  0.0994,
         -0.0147, -0.0780, -0.0333,  0.1444,  0.1170, -0.0851,  0.1472, -0.1025],
        [-0.0159, -0.0256,  0.0612, -0.1095,  0.0327,  0.1125, -0.0148,  0.0808,
         -0.0120, -0.0707, -0.0299,  0.1410,  0.1264, -0.0773,  0.1461, -0.1134]],
       grad_fn=<AddmmBackward0>)

In [8]:
#参数访问
net = nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net[2].state_dict()['weight'].grad==None


True

In [9]:
print(*[(name,para.shape) for name,para in net[0].named_parameters()])
print(*[(name,para.shape) for name,para in net.named_parameters()])
print(net[0].state_dict()["weight"].shape)
print(net.state_dict()['2.weight'].shape)

('weight', torch.Size([256, 20])) ('bias', torch.Size([256]))
('0.weight', torch.Size([256, 20])) ('0.bias', torch.Size([256])) ('2.weight', torch.Size([10, 256])) ('2.bias', torch.Size([10]))
torch.Size([256, 20])
torch.Size([10, 256])


In [10]:
# Sequential支持嵌套层

In [11]:
#初始化参数
def init_normal(m):
    if type(m)==nn.Linear:
        nn.init.normal_(m.weight,mean=0,std=0.01)
        nn.init.zeros_(m.bias)

net.apply(init_normal)

def init_constant(m):
    if type(m)==nn.Linear:
        nn.init.constant_(m.weight,1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)

def init_xavier(m):
    if type(m)==nn.Linear:
        nn.init.xavier_uniform_(m.weight)

net.apply(init_xavier) 

net[0].weight.data[:]+=1

In [12]:
#参数绑定,两层共享权重时
shared=nn.Linear(8,8)
net=nn.Sequential(nn.Linear(4,8),nn.ReLU(),shared,nn.ReLU(),shared,nn.ReLU(),nn.Linear(8,1))

In [13]:
#定义层：
class CenterdLayer(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,X):
        return X-X.mean()

x=torch.randn(4,8)
layer=CenterdLayer()
layer(x)

#定义含参层
class MyLinear(nn.Module):
    def __init__(self,in_units,out_units):
        super().__init__()
        self.weight=nn.Parameter(torch.randn(in_units,out_units))
        self.bias=nn.Parameter(torch.randn(out_units))

    def forward(self,X):
        linear= torch.matmul(X,self.weight.data)+self.bias.data
        return linear

In [ ]:
#文件读写：支持类型：字典、张量、列表嵌套列表（python基础类型+派生类型）- 注意如何读写和存储整个模型的所有参数的集合。
x=torch.randn(2,3)
y=torch.rand(2,3)
z=torch.arange(4)
x,y,z

torch.save(x,"x_file.checkpoint")
x2=torch.load('x_file.checkpoint')
torch.save([x,y],"xy_file.checkpoint")

mydict={'x':x,'y':y,'z':z}
torch.save(mydict,'mydict_file.checkpoint')

#存储参数
torch.save(net.state_dict(),'net_params.pth')

#加载参数
clone=nn.Sequential(nn.Linear(4,8),nn.ReLU(),shared,nn.ReLU(),shared,nn.ReLU(),nn.Linear(8,1))
clone.load_state_dict(torch.load('net_params.pth'))
clone.eval()


Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=8, bias=True)
  (5): ReLU()
  (6): Linear(in_features=8, out_features=1, bias=True)
)

In [19]:
#尝试不同的设备进行计算
torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.device('cuda')
torch.device('cpu')
torch.device('cuda:0')

def try_gpu(i=0):
    return torch.device(f"cuda:{i}" if torch.cuda.device_count()>i else 'cpu')

x=torch.tensor([1,2,3],dtype=torch.float32,device=torch.device('cuda:0'))
y=torch.tensor([1,2,3],device=try_gpu(1))
z=x.cuda(0)   #只有存储在同一个设备上的tensor可以运算

#神经网络的设备
net=nn.Sequential(nn.Linear(3,1))
net=net.to(device=try_gpu(0))

net(x)

net[0].weight.data.device

device(type='cuda', index=0)